# CNN Spatial Appearance Stream — Deepfake Detection

## Two-Stream Late Fusion Architecture (Stream 1 of 2)

This notebook trains the **spatial appearance stream** using EfficientNet-B4 to detect deepfake generation artifacts.

```
┌─────────────────────────────────────────────────────────────┐
│                    LATE FUSION ARCHITECTURE                  │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   Stream 1 (THIS NOTEBOOK)         Stream 2 (rPPG)          │
│   ────────────────────────         ─────────────────        │
│   Video → Frames → Faces           Video → rPPG Features    │
│         ↓                                  ↓                │
│   EfficientNet-B4                  ML Classifier            │
│         ↓                                  ↓                │
│      P_CNN                              P_rPPG              │
│         \                                /                  │
│          \                              /                   │
│           \                            /                    │
│            └──────── FUSION ──────────┘                     │
│                       ↓                                     │
│               Final Prediction                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**Output:** `cnn_predictions.csv` with video-level P_CNN scores

## 1. Setup & Imports

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess
import sys

def install_packages():
    packages = [
        "facenet-pytorch",
        "timm",
        "albumentations",
        "opencv-python-headless",
    ]
    for pkg in packages:
        print(f"Installing {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    print("✓ All packages installed")

install_packages()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import os
import gc
import cv2
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import List, Tuple, Dict, Optional
from tqdm.auto import tqdm
import io
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# PyTorch 2.0+ compatible AMP imports
try:
    from torch.amp import autocast, GradScaler
    USE_NEW_AMP = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    USE_NEW_AMP = False

import timm
from facenet_pytorch import MTCNN
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

warnings.filterwarnings('ignore')

# ─── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ─── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

class Config:
    # Dataset paths
    DATASET_ROOT = "/kaggle/input/datasets/likhithvasireddy/400videoseach/content/drive/MyDrive/face_dataset_dip"
    REAL_DIR = os.path.join(DATASET_ROOT, "real_videos")
    FAKE_DIR = os.path.join(DATASET_ROOT, "deepfake_videos")
    OUTPUT_DIR = "/kaggle/working"
    
    # Frame extraction
    FRAMES_PER_VIDEO = 15
    IMG_SIZE = 224
    
    # Training
    BATCH_SIZE = 32
    NUM_EPOCHS = 15
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    NUM_WORKERS = 2
    
    # Model
    MODEL_NAME = "efficientnet_b4"
    DROPOUT = 0.4
    HIDDEN_DIM = 256
    
    # Split
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.2

cfg = Config()

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("="*70)
print("CNN SPATIAL STREAM CONFIGURATION")
print("="*70)
print(f"  Dataset root: {cfg.DATASET_ROOT}")
print(f"  Frames per video: {cfg.FRAMES_PER_VIDEO}")
print(f"  Image size: {cfg.IMG_SIZE}x{cfg.IMG_SIZE}")
print(f"  Batch size: {cfg.BATCH_SIZE}")
print(f"  Epochs: {cfg.NUM_EPOCHS}")
print(f"  Learning rate: {cfg.LEARNING_RATE}")
print(f"  Model: {cfg.MODEL_NAME}")
print("="*70)

## 2. Frame Extraction & Face Detection

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MTCNN FACE DETECTOR
# ═══════════════════════════════════════════════════════════════════════════════

class FaceExtractor:
    """
    Extracts faces from video frames using MTCNN.
    Falls back to center crop if no face is detected.
    """
    
    def __init__(self, device, img_size=224, margin=40):
        self.device = device
        self.img_size = img_size
        self.margin = margin
        
        # Initialize MTCNN with optimized settings
        self.mtcnn = MTCNN(
            image_size=img_size,
            margin=margin,
            min_face_size=60,
            thresholds=[0.6, 0.7, 0.7],
            factor=0.709,
            post_process=False,  # Raw pixel values [0, 255], no [-1, 1] normalization
            device=device,
            keep_all=False,  # Only largest face
        )
        print(f"✓ MTCNN initialized on {device}")
    
    def extract_face(self, frame: np.ndarray) -> Optional[np.ndarray]:
        """
        Extract face from a single frame.
        Returns: Face crop as numpy array (H, W, 3) or None if failed.
        """
        try:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # Detect face
            face = self.mtcnn(pil_img)
            
            if face is not None:
                # MTCNN returns tensor (C, H, W), convert to numpy (H, W, C)
                # Since post_process=False, values are already 0-255
                face_np = face.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
                return face_np
            else:
                # Fallback: center crop
                return self._center_crop(frame_rgb)
                
        except Exception as e:
            # Fallback: center crop
            return self._center_crop(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    def _center_crop(self, frame: np.ndarray) -> np.ndarray:
        """Center crop fallback when face detection fails."""
        h, w = frame.shape[:2]
        size = min(h, w)
        y = (h - size) // 2
        x = (w - size) // 2
        crop = frame[y:y+size, x:x+size]
        return cv2.resize(crop, (self.img_size, self.img_size))


# Initialize face extractor
face_extractor = FaceExtractor(DEVICE, img_size=cfg.IMG_SIZE)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO FRAME EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════

def extract_frames_from_video(video_path: str, n_frames: int = 10) -> List[np.ndarray]:
    """
    Extract n evenly spaced frames from a video.
    
    Args:
        video_path: Path to video file
        n_frames: Number of frames to extract
        
    Returns:
        List of frame arrays (BGR format)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return []
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return []  # Corrupted video
    
    if total_frames < n_frames:
        # If video has fewer frames, extract all
        frame_indices = list(range(total_frames))
    else:
        # Evenly spaced frame indices
        frame_indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    
    cap.release()
    return frames


def process_video(video_path: str, face_extractor: FaceExtractor, n_frames: int = 10) -> List[np.ndarray]:
    """
    Extract frames and detect faces from a video.
    
    Returns:
        List of face crops as numpy arrays
    """
    frames = extract_frames_from_video(video_path, n_frames)
    
    face_crops = []
    for frame in frames:
        face = face_extractor.extract_face(frame)
        if face is not None:
            face_crops.append(face)
    
    return face_crops


print("✓ Frame extraction functions defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROCESS ALL VIDEOS
# ═══════════════════════════════════════════════════════════════════════════════

def collect_video_paths():
    """Collect all video paths with labels."""
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    
    videos = []
    
    # Real videos (label = 0)
    if os.path.exists(cfg.REAL_DIR):
        for f in sorted(os.listdir(cfg.REAL_DIR)):
            if f.lower().endswith(video_extensions):
                videos.append({
                    'video_id': f,
                    'path': os.path.join(cfg.REAL_DIR, f),
                    'label': 0  # Real
                })
    
    # Fake videos (label = 1)
    if os.path.exists(cfg.FAKE_DIR):
        for f in sorted(os.listdir(cfg.FAKE_DIR)):
            if f.lower().endswith(video_extensions):
                videos.append({
                    'video_id': f,
                    'path': os.path.join(cfg.FAKE_DIR, f),
                    'label': 1  # Fake
                })
    
    return videos

# Collect videos
all_videos = collect_video_paths()
print(f"Total videos found: {len(all_videos)}")
print(f"  Real: {sum(1 for v in all_videos if v['label'] == 0)}")
print(f"  Fake: {sum(1 for v in all_videos if v['label'] == 1)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXTRACT ALL FACES (with caching)
# ═══════════════════════════════════════════════════════════════════════════════

def extract_all_faces(videos: List[dict], face_extractor: FaceExtractor) -> Dict[str, List[np.ndarray]]:
    """
    Extract faces from all videos.
    
    Returns:
        Dictionary mapping video_id to list of face crops
    """
    face_data = {}
    
    print("\n" + "="*70)
    print("EXTRACTING FACES FROM VIDEOS")
    print("="*70)
    
    failed_videos = []
    
    for video in tqdm(videos, desc="Processing videos"):
        video_id = video['video_id']
        video_path = video['path']
        
        faces = process_video(video_path, face_extractor, cfg.FRAMES_PER_VIDEO)
        
        if len(faces) >= 3:  # Require at least 3 valid faces
            face_data[video_id] = faces
        else:
            failed_videos.append(video_id)
    
    print(f"\n✓ Successfully processed: {len(face_data)} videos")
    print(f"✗ Failed/insufficient faces: {len(failed_videos)} videos")
    
    # Memory cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return face_data

# Extract faces
face_data = extract_all_faces(all_videos, face_extractor)

# DO NOT shrink all_videos — this would misalign train_test_split indices
# with the ML notebook. Downstream code (Dataset, predict) already checks
# `if video_id in face_data` to safely skip missing videos.

# Validate we have face data
if len(face_data) == 0:
    raise ValueError("ERROR: No videos with valid face detections! Check face extraction.")
print(f"\nVideos with valid faces: {len(face_data)}/{len(all_videos)}")

## 3. Dataset Creation & Data Leakage Prevention

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAIN/VAL SPLIT BY VIDEO (CRITICAL: NO DATA LEAKAGE)
# ═══════════════════════════════════════════════════════════════════════════════

def split_videos_by_id(videos: List[dict], train_ratio: float = 0.8):
    """
    Split videos into train/val sets ensuring no video appears in both.
    Uses a SINGLE stratified split to match the ML pipeline's pseudo-random sequence.
    
    CRITICAL: Frames from the same video NEVER leak across splits.
    CRITICAL: Must match ML notebook's train_test_split(X, y, test_size=0.2, 
              random_state=42, stratify=y) sequence for Late Fusion alignment.
    """
    labels = [v['label'] for v in videos]
    
    train_videos, val_videos = train_test_split(
        videos,
        train_size=train_ratio,
        random_state=SEED,
        stratify=labels
    )
    
    return train_videos, val_videos


# Split videos
train_videos, val_videos = split_videos_by_id(all_videos, cfg.TRAIN_RATIO)

print("\n" + "="*70)
print("TRAIN/VALIDATION SPLIT (BY VIDEO ID - NO LEAKAGE)")
print("="*70)
print(f"\nTrain videos: {len(train_videos)}")
print(f"  Real: {sum(1 for v in train_videos if v['label'] == 0)}")
print(f"  Fake: {sum(1 for v in train_videos if v['label'] == 1)}")
print(f"\nVal videos: {len(val_videos)}")
print(f"  Real: {sum(1 for v in val_videos if v['label'] == 0)}")
print(f"  Fake: {sum(1 for v in val_videos if v['label'] == 1)}")

# Verify no leakage
train_ids = set(v['video_id'] for v in train_videos)
val_ids = set(v['video_id'] for v in val_videos)
assert len(train_ids & val_ids) == 0, "DATA LEAKAGE DETECTED!"
print("\n✓ NO DATA LEAKAGE: Train and Val sets are completely separate")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATA AUGMENTATION
# ═══════════════════════════════════════════════════════════════════════════════

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_train_transforms():
    """Training augmentations optimized for deepfake detection."""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=10, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.ImageCompression(quality_lower=50, quality_upper=100, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.CoarseDropout(max_holes=4, max_height=20, max_width=20, p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_val_transforms():
    """Validation transforms: only normalization."""
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


print("✓ Augmentation pipelines defined")
print("  Train: HorizontalFlip, RandomBrightness, JPEG Compression, GaussNoise")
print("  Val: Normalize only")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PYTORCH DATASET
# ═══════════════════════════════════════════════════════════════════════════════

class DeepfakeFaceDataset(Dataset):
    """
    PyTorch Dataset for face-level deepfake detection.
    Each sample is a single face crop with its video-level label.
    """
    
    def __init__(self, videos: List[dict], face_data: Dict[str, List[np.ndarray]], 
                 transform=None, is_train=True):
        """
        Args:
            videos: List of video dicts with 'video_id' and 'label'
            face_data: Dict mapping video_id to list of face crops
            transform: Albumentations transform pipeline
            is_train: Whether this is training set
        """
        self.transform = transform
        self.is_train = is_train
        
        # Build flat list of (face_crop, label, video_id)
        self.samples = []
        for video in videos:
            video_id = video['video_id']
            label = video['label']
            
            if video_id in face_data:
                for face in face_data[video_id]:
                    self.samples.append((face, label, video_id))
        
        print(f"  Dataset created: {len(self.samples)} samples from {len(videos)} videos")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        face, label, video_id = self.samples[idx]
        
        # Apply transforms
        if self.transform:
            augmented = self.transform(image=face)
            face = augmented['image']
        
        return {
            'image': face,
            'label': torch.tensor(label, dtype=torch.float32),
            'video_id': video_id
        }


# Create datasets
print("\nCreating datasets...")
train_dataset = DeepfakeFaceDataset(
    train_videos, face_data, transform=get_train_transforms(), is_train=True
)
val_dataset = DeepfakeFaceDataset(
    val_videos, face_data, transform=get_val_transforms(), is_train=False
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, 
    num_workers=cfg.NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
    num_workers=cfg.NUM_WORKERS, pin_memory=True
)

print(f"\n✓ Train loader: {len(train_loader)} batches")
print(f"✓ Val loader: {len(val_loader)} batches")

## 4. CNN Model Architecture

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EFFICIENTNET-B4 MODEL
# ═══════════════════════════════════════════════════════════════════════════════

class DeepfakeCNN(nn.Module):
    """
    EfficientNet-B4 based deepfake detector.
    
    Architecture:
        EfficientNet-B4 (pretrained) → GlobalAvgPool → Dense(256) → Dropout(0.4) → Dense(1)
    
    Note: No Sigmoid layer - using BCEWithLogitsLoss for numerical stability.
    """
    
    def __init__(self, model_name='efficientnet_b4', hidden_dim=256, dropout=0.4, pretrained=True):
        super().__init__()
        
        # Load pretrained EfficientNet-B4
        self.backbone = timm.create_model(
            model_name, 
            pretrained=pretrained,
            num_classes=0,  # Remove classifier, keep features
            global_pool='avg'  # Global average pooling
        )
        
        # Get feature dimension
        backbone_dim = self.backbone.num_features
        
        # Custom classifier head
        self.classifier = nn.Sequential(
            nn.Linear(backbone_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)  # No Sigmoid - using BCEWithLogitsLoss
        )
        
        # Initialize classifier weights
        self._init_weights()
        
        print(f"\n✓ {model_name} loaded")
        print(f"  Backbone features: {backbone_dim}")
        print(f"  Hidden dim: {hidden_dim}")
        print(f"  Dropout: {dropout}")
    
    def _init_weights(self):
        """Initialize classifier weights."""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        features = self.backbone(x)  # (B, backbone_dim)
        logits = self.classifier(features)  # (B, 1)
        return logits.squeeze(-1)  # (B,)


# Create model
model = DeepfakeCNN(
    model_name=cfg.MODEL_NAME,
    hidden_dim=cfg.HIDDEN_DIM,
    dropout=cfg.DROPOUT,
    pretrained=True
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Training Protocol

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """
    Train for one epoch with AMP (Automatic Mixed Precision).
    """
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Training", leave=False)
    
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        # AMP forward pass
        with autocast(device_type=str(device).split(":")[0]) if USE_NEW_AMP else autocast():
            logits = model(images)
            loss = criterion(logits, labels)
        
        # AMP backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        # Track metrics
        total_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(loader.dataset)
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = accuracy_score(all_labels, (all_preds > 0.5).astype(int))
    auc = roc_auc_score(all_labels, all_preds)
    
    return avg_loss, acc, auc


@torch.no_grad()
def validate(model, loader, criterion, device):
    """
    Validate the model.
    """
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(loader, desc="Validating", leave=False):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        with autocast(device_type=str(device).split(":")[0]) if USE_NEW_AMP else autocast():
            logits = model(images)
            loss = criterion(logits, labels)
        
        total_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = accuracy_score(all_labels, (all_preds > 0.5).astype(int))
    auc = roc_auc_score(all_labels, all_preds)
    f1 = f1_score(all_labels, (all_preds > 0.5).astype(int))
    
    return avg_loss, acc, auc, f1


print("✓ Training utilities defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════════

# Loss, optimizer, scheduler
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=cfg.LEARNING_RATE, 
    weight_decay=cfg.WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6
)

# AMP scaler for mixed precision
scaler = GradScaler("cuda") if (USE_NEW_AMP and torch.cuda.is_available()) else GradScaler()

# Training history
history = {
    'train_loss': [], 'train_acc': [], 'train_auc': [],
    'val_loss': [], 'val_acc': [], 'val_auc': [], 'val_f1': []
}

best_val_auc = 0.0
best_epoch = 0
patience = 5
patience_counter = 0

print("\n" + "="*70)
print("TRAINING CNN SPATIAL STREAM")
print("="*70)
print(f"  Epochs: {cfg.NUM_EPOCHS}")
print(f"  Batch size: {cfg.BATCH_SIZE}")
print(f"  Learning rate: {cfg.LEARNING_RATE}")
print(f"  Mixed Precision: Enabled (AMP)")
print("="*70 + "\n")

for epoch in range(cfg.NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{cfg.NUM_EPOCHS}")
    print("-" * 40)
    
    # Train
    train_loss, train_acc, train_auc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE
    )
    
    # Validate
    val_loss, val_acc, val_auc, val_f1 = validate(
        model, val_loader, criterion, DEVICE
    )
    
    # Update scheduler
    scheduler.step()
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['val_f1'].append(val_f1)
    
    # Print metrics
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f} | F1: {val_f1:.4f}")
    
    # Save best model + early stopping
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(cfg.OUTPUT_DIR, "best_cnn_model.pth"))
        print(f"  ★ New best model saved (AUC: {val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"  Best Epoch: {best_epoch}")
print(f"  Best Val AUC: {best_val_auc:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING CURVES
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['val_loss'], label='Val', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', marker='o')
axes[1].plot(history['val_acc'], label='Val', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# AUC
axes[2].plot(history['train_auc'], label='Train', marker='o')
axes[2].plot(history['val_auc'], label='Val', marker='s')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].set_title('AUC Curve')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Video-Level Inference & Export

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD BEST MODEL
# ═══════════════════════════════════════════════════════════════════════════════

# Load best model weights
model.load_state_dict(torch.load(os.path.join(cfg.OUTPUT_DIR, "best_cnn_model.pth"), map_location=DEVICE, weights_only=True))
model.eval()
print("✓ Best model loaded for inference")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO-LEVEL PREDICTION
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def predict_video(model, faces: List[np.ndarray], transform, device) -> float:
    """
    Predict deepfake probability for a single video.
    Averages predictions across all face frames.
    
    Returns:
        P_CNN: Average sigmoid probability (0 = real, 1 = fake)
    """
    model.eval()
    
    if len(faces) == 0:
        return 0.5  # Neutral if no faces
    
    probs = []
    
    for face in faces:
        # Apply transforms
        augmented = transform(image=face)
        face_tensor = augmented['image'].unsqueeze(0).to(device)
        
        # Forward pass
        with autocast(device_type=str(device).split(":")[0]) if USE_NEW_AMP else autocast():
            logit = model(face_tensor)
        
        prob = torch.sigmoid(logit).item()
        probs.append(prob)
    
    # Average across frames
    return np.mean(probs)


def generate_video_predictions(model, all_videos, face_data, device):
    """
    Generate video-level predictions for all videos.
    
    Returns:
        DataFrame with video_id and P_CNN columns
    """
    transform = get_val_transforms()
    
    results = []
    
    print("\n" + "="*70)
    print("GENERATING VIDEO-LEVEL PREDICTIONS")
    print("="*70)
    
    for video in tqdm(all_videos, desc="Predicting"):
        video_id = video['video_id']
        
        if video_id in face_data:
            faces = face_data[video_id]
            p_cnn = predict_video(model, faces, transform, device)
        else:
            p_cnn = 0.5  # Neutral if no face data
        
        results.append({
            'video_id': video_id,
            'P_CNN': p_cnn
        })
    
    return pd.DataFrame(results)


# Generate predictions
predictions_df = generate_video_predictions(model, all_videos, face_data, DEVICE)

print(f"\n✓ Predictions generated for {len(predictions_df)} videos")
print(predictions_df.head(10))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT PREDICTIONS
# ═══════════════════════════════════════════════════════════════════════════════

# Add ground truth labels
video_labels = {v['video_id']: v['label'] for v in all_videos}
predictions_df['true_label'] = predictions_df['video_id'].map(video_labels)

# Save predictions CSV (with true_label to match rPPG export format)
csv_path = os.path.join(cfg.OUTPUT_DIR, "cnn_predictions.csv")
predictions_df.to_csv(csv_path, index=False)
print(f"\n✓ Predictions saved to: {csv_path}")

# Show distribution
print("\nP_CNN Distribution:")
print(predictions_df['P_CNN'].describe())

# Alias for metrics below
predictions_df['label'] = predictions_df['true_label']

# Calculate video-level metrics
y_true = predictions_df['label'].values
y_pred_proba = predictions_df['P_CNN'].values
y_pred = (y_pred_proba > 0.5).astype(int)

video_acc = accuracy_score(y_true, y_pred)
# Handle edge case where only one class is present
if len(np.unique(y_true)) > 1:
    video_auc = roc_auc_score(y_true, y_pred_proba)
else:
    video_auc = 0.5
    print("Warning: Only one class in predictions, AUC set to 0.5")
video_f1 = f1_score(y_true, y_pred)

print("\n" + "="*70)
print("VIDEO-LEVEL METRICS (Full Dataset)")
print("="*70)
print(f"  Accuracy: {video_acc:.4f}")
print(f"  AUC-ROC:  {video_auc:.4f}")
print(f"  F1-Score: {video_f1:.4f}")
print("="*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZE PREDICTION DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution by class
real_preds = predictions_df[predictions_df['label'] == 0]['P_CNN']
fake_preds = predictions_df[predictions_df['label'] == 1]['P_CNN']

axes[0].hist(real_preds, bins=30, alpha=0.6, label='Real', color='green', density=True)
axes[0].hist(fake_preds, bins=30, alpha=0.6, label='Fake', color='red', density=True)
axes[0].axvline(x=0.5, color='black', linestyle='--', label='Threshold')
axes[0].set_xlabel('P_CNN')
axes[0].set_ylabel('Density')
axes[0].set_title('P_CNN Distribution by Class')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
axes[1].plot(fpr, tpr, 'b-', label=f'CNN (AUC = {video_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (Video-Level)')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'cnn_results.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ALL ARTIFACTS
# ═══════════════════════════════════════════════════════════════════════════════

# Save final model weights
torch.save(model.state_dict(), os.path.join(cfg.OUTPUT_DIR, "cnn_spatial_stream_final.pth"))

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(cfg.OUTPUT_DIR, "cnn_training_history.csv"), index=False)

# Save config
config_dict = {
    'model_name': cfg.MODEL_NAME,
    'hidden_dim': cfg.HIDDEN_DIM,
    'dropout': cfg.DROPOUT,
    'img_size': cfg.IMG_SIZE,
    'frames_per_video': cfg.FRAMES_PER_VIDEO,
    'batch_size': cfg.BATCH_SIZE,
    'learning_rate': cfg.LEARNING_RATE,
    'num_epochs': cfg.NUM_EPOCHS,
    'best_val_auc': best_val_auc,
    'video_level_auc': video_auc,
}
pd.DataFrame([config_dict]).to_csv(os.path.join(cfg.OUTPUT_DIR, "cnn_config.csv"), index=False)

print("\n" + "="*70)
print("ALL ARTIFACTS SAVED")
print("="*70)
print(f"  ✓ cnn_predictions.csv        (video-level P_CNN scores)")
print(f"  ✓ best_cnn_model.pth          (best model weights)")
print(f"  ✓ cnn_spatial_stream_final.pth (final model weights)")
print(f"  ✓ cnn_training_history.csv   (training metrics)")
print(f"  ✓ cnn_config.csv             (configuration)")
print(f"  ✓ training_curves.png        (loss/acc/auc plots)")
print(f"  ✓ cnn_results.png            (prediction distribution)")
print("="*70)

## 7. Late Fusion Integration Guide

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LATE FUSION INTEGRATION GUIDE
# ═══════════════════════════════════════════════════════════════════════════════

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                       LATE FUSION INTEGRATION GUIDE                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  This notebook outputs: cnn_predictions.csv                                  ║
║  Columns: video_id, P_CNN                                                    ║
║                                                                              ║
║  To fuse with rPPG predictions from 2ND_MODEL.ipynb:                        ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────────┐ ║
║  │  # Load both predictions                                                │ ║
║  │  cnn_df = pd.read_csv('cnn_predictions.csv')                           │ ║
║  │  rppg_df = pd.read_csv('rppg_predictions.csv')                         │ ║
║  │                                                                         │ ║
║  │  # Merge on video_id                                                    │ ║
║  │  fused_df = cnn_df.merge(rppg_df, on='video_id')                       │ ║
║  │                                                                         │ ║
║  │  # Simple average fusion                                                │ ║
║  │  fused_df['P_final'] = (fused_df['P_CNN'] + fused_df['P_rPPG']) / 2    │ ║
║  │                                                                         │ ║
║  │  # Weighted fusion (if CNN is more accurate)                           │ ║
║  │  w_cnn, w_rppg = 0.6, 0.4                                              │ ║
║  │  fused_df['P_final'] = w_cnn * fused_df['P_CNN']                       │ ║
║  │                      + w_rppg * fused_df['P_rPPG']                      │ ║
║  │                                                                         │ ║
║  │  # Learned fusion (train a small classifier)                           │ ║
║  │  from sklearn.linear_model import LogisticRegression                   │ ║
║  │  X_fusion = fused_df[['P_CNN', 'P_rPPG']].values                       │ ║
║  │  y_fusion = fused_df['label'].values                                   │ ║
║  │  fusion_model = LogisticRegression().fit(X_fusion, y_fusion)           │ ║
║  │  fused_df['P_final'] = fusion_model.predict_proba(X_fusion)[:, 1]      │ ║
║  └─────────────────────────────────────────────────────────────────────────┘ ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("CNN SPATIAL STREAM — FINAL SUMMARY")
print("="*70)
print(f"\n  Model: EfficientNet-B4")
print(f"  Total Parameters: {total_params:,}")
print(f"  Training Samples: {len(train_dataset)}")
print(f"  Validation Samples: {len(val_dataset)}")
print(f"\n  Best Validation AUC: {best_val_auc:.4f}")
print(f"  Video-Level AUC: {video_auc:.4f}")
print(f"  Video-Level Accuracy: {video_acc:.4f}")
print(f"  Video-Level F1: {video_f1:.4f}")
print(f"\n  Output: {csv_path}")
print("="*70)
print("\n✅ CNN Spatial Stream training complete!")
print("   Ready for Late Fusion with rPPG stream.")